# 05 · Chronos-2, zero-shot and LoRA fine-tuned, on the notebook-02 output

Same data, blocks, folds, origins and metrics as notebooks 03 and 04, so the results line up.
One pooled Chronos-2 serves every device and both targets (each device and target is one
series), with the calendar and the site weather at the target hour as covariates. Zero-shot
reads the CONTEXT_LENGTH hours before each origin; the fine-tuned model adds LoRA adapters
trained on labels before the block: a small grid is scored on the tuning folds (trained before
TUNE_START), and the winner is refitted before TEST_START for the test folds and before CUTOFF
for the frozen live replay. Every setting lives in the first code cell, tagged `parameters` for
papermill; it needs a GPU, so the last cell runs it on the GX10.

In [ ]:
# What to forecast
TARGETS = ["grid_export", "grid_import"]   # grid_export only runs on devices with PV
HORIZON = 168                              # every origin forecasts hours 1..HORIZON
DEVICES = None                             # None: every device; or a list of ids
# Weather is read at the target hour, so the results are an upper bound until as-issued
# weather forecasts land in notebook 01.
WEATHER = "perfect"                        # the only value for now

# Blocks, local dates (Europe/Rome): tune [TUNE_START, TEST_START), test [TEST_START, CUTOFF),
# then a live replay of the model frozen at CUTOFF
TUNE_START = "2026-06-01"
TEST_START = "2026-08-01"
CUTOFF = "2026-09-01"
TUNE_FOLD_DAYS = 14
TEST_FOLD_DAYS = 7
ORIGIN_STRIDE_H = 24                       # tune/test origins from local midnight
REPLAY_STRIDE_H = 6                        # replay origins at local 00/06/12/18 from CUTOFF

# Chronos-2: one pooled model, native quantiles, point = median
MODEL_ID = "amazon/chronos-2"
CONTEXT_LENGTH = 512                       # zero-shot history in hours, up to 8192
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
PRED_BATCH_SIZE = 256                      # series (targets + covariates) per forward pass
# LoRA fine-tuning (r=8, alpha=16) grid of (steps, learning rate, context hours), scored on the
# tuning folds; the best setting is refitted for test and replay. The context is used both per
# training window and per forecast.
FT_GRID = [(1000, 1e-4, 512), (250, 1e-4, 512), (1000, 3e-5, 512), (1000, 1e-4, 2048)]
FT_BATCH_SIZE = 64                         # series per training step (8 per item: 1 + 7 covariates)
GPU_MEMORY_FRACTION = 0.12                 # of the GX10's ~121 GB unified memory (Qwen holds ~65)
SEED = 42

# Output
RUN_NAME = None                            # None: "chronos2-h<HORIZON>-<timestamp>"
LOG_MLFLOW = True                          # MLFLOW_TRACKING_URI if set, else runs/mlflow.db
MLFLOW_EXPERIMENT = "nb05-chronos2"

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

PARAMS = {k: v for k, v in dict(globals()).items() if k.isupper()}
assert WEATHER == "perfect", "only perfect-forecast weather is available so far"
assert {0.1, 0.25, 0.5, 0.75, 0.9} <= set(QUANTILES), "the median and 50%/80% intervals need these"
assert HORIZON <= 168, "naive168 would read labels after the origin"
FT_GRID = [dict(zip(["steps", "lr", "context"], g, strict=True)) for g in FT_GRID]

NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
DATA_DIR = NB_DIR / "data"
RUN_NAME = RUN_NAME or f"chronos2-h{HORIZON}-{time.strftime('%Y%m%d-%H%M%S')}"
OUT_DIR = REPO_ROOT / "runs" / "nb05" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

PT = pd.read_parquet(DATA_DIR / "processed_hourly.parquet")
quality = pd.read_csv(DATA_DIR / "device_quality.csv").set_index("device_id")
HAS_PV = quality["has_pv"].astype(bool).to_dict()
ALL_DEVICES = sorted(DEVICES or PT["device_id"].unique())
PT = PT[PT["device_id"].isin(ALL_DEVICES)]
# One continuous hourly frame per device, so a lag is a plain index lookup.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}
LOCAL_TZ = "Europe/Rome"
print("run:", RUN_NAME, "| devices:", len(ALL_DEVICES), "| rows:", len(PT), "| output:", OUT_DIR)

In [ ]:
# Origins as UTC timestamps. ts_hour labels the start of the hour: a forecast issued at clock
# time `origin` has seen labels up to origin - 1h and returns labels origin .. origin +
# (HORIZON - 1)h, lead h = 1..HORIZON. A path counts only if its last label is before the
# block end (tune/test) or has been observed (replay).
T_TUNE, T_TEST, CUT = (pd.Timestamp(d, tz=LOCAL_TZ) for d in (TUNE_START, TEST_START, CUTOFF))
PATH_H = pd.Timedelta(hours=HORIZON)
rows = []
for block, start, end, days in [("tune", T_TUNE, T_TEST, TUNE_FOLD_DAYS),
                                ("test", T_TEST, CUT, TEST_FOLD_DAYS)]:
    for fold in pd.date_range(start, end, freq=f"{days}D", inclusive="left"):
        stop = min(fold + pd.Timedelta(days=days), end)
        rows += [(block, fold, o) for o in pd.date_range(fold, stop, freq=f"{ORIGIN_STRIDE_H}h",
                                                         inclusive="left") if o + PATH_H <= end]
ORIGINS = pd.DataFrame(rows, columns=["block", "fold", "origin"])
ORIGINS[["fold", "origin"]] = ORIGINS[["fold", "origin"]].apply(lambda c: c.dt.tz_convert("UTC"))
T_TUNE, T_TEST, CUT = (t.tz_convert("UTC") for t in (T_TUNE, T_TEST, CUT))
BLOCK_END = {"tune": T_TEST, "test": CUT}
PAIRS = [(d, t) for t in TARGETS for d in ALL_DEVICES if t != "grid_export" or HAS_PV.get(d, False)]

# Every scored row: its actual, the two naive forecasts and is_daylight, keyed by
# (device, target, origin, horizon); rows without an actual are dropped.
parts = []
for d, target in PAIRS:
    s = DEV[d][target]
    last = DEV[d].index[-1] - PATH_H + pd.Timedelta(hours=1)
    replay = pd.date_range(CUT, last, freq=f"{REPLAY_STRIDE_H}h")
    orig = pd.concat([ORIGINS, pd.DataFrame({"block": "replay", "fold": CUT, "origin": replay})],
                     ignore_index=True)
    h = np.tile(np.arange(1, HORIZON + 1), len(orig))
    o = pd.DatetimeIndex(orig["origin"]).repeat(HORIZON)
    t = o + pd.to_timedelta(h - 1, unit="h")
    lag168 = t - pd.Timedelta(hours=168)
    lag24 = t - pd.to_timedelta(24 * np.ceil(h / 24), unit="h")  # most recent observable day
    meta = pd.DataFrame({
        "device_id": d, "target": target, "block": orig["block"].repeat(HORIZON).array,
        "fold": orig["fold"].repeat(HORIZON).array, "origin": o, "horizon": h, "ts_hour": t,
        "actual": s.reindex(t).to_numpy(), "naive168": s.reindex(lag168).to_numpy(),
        "naive24": s.reindex(lag24).to_numpy(),
        "is_daylight": DEV[d]["is_daylight"].reindex(t).to_numpy()})
    assert (meta.groupby("origin").size() == HORIZON).all()  # full paths, before dropping gaps
    assert (lag168 < o).all() and (lag24 < o).all()  # the naives read before the origin
    assert ((replay - CUT) % pd.Timedelta(hours=REPLAY_STRIDE_H) == pd.Timedelta(0)).all()
    parts.append(meta[meta["actual"].notna()])
META = pd.concat(parts, ignore_index=True)
for block, end in BLOCK_END.items():  # no tuning or test actual at or after its block end
    assert (META.loc[META["block"] == block, "ts_hour"] < end).all()
first = META.loc[META["block"] == "replay", "ts_hour"].min()
assert first == CUT, "the first replay target label is CUTOFF"
# One forecast per (device, target, origin) that has at least one actual.
KEY = ["device_id", "target", "origin", "horizon"]
PATHS = META.drop_duplicates(["block", *KEY[:3]])[["block", *KEY[:3]]]
print(f"scored rows {len(META):,} | paths {PATHS.groupby('block').size().to_dict()}"
      f" | replay from {first.tz_convert(LOCAL_TZ)}")

In [ ]:
# Series and covariates on one hourly UTC grid, so a context is a plain array slice. The
# covariates are known at the target hour: local hour and day of week as sin/cos, the holiday
# flag, and the site weather (perfect forecast, gaps interpolated).
import gc

import torch
from chronos import Chronos2Pipeline
from transformers import set_seed

HOLIDAYS = {pd.Timestamp(d).date() for d in [
    "2025-01-01", "2025-01-06", "2025-04-21", "2025-04-25", "2025-05-01", "2025-06-02",
    "2025-08-15", "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-06", "2026-04-25", "2026-05-01", "2026-06-02",
    "2026-08-15", "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26"]}
WEATHER_COLS = ["global_tilted_irradiance", "temperature_2m", "cloud_cover"]
MAX_CTX = max([CONTEXT_LENGTH, *(g["context"] for g in FT_GRID)])
HOURS = pd.date_range(PT["ts_hour"].min() - pd.Timedelta(hours=MAX_CTX),
                      PT["ts_hour"].max() + PATH_H, freq="h")
local = HOURS.tz_convert(LOCAL_TZ)
COV = pd.DataFrame({"hour_sin": np.sin(2 * np.pi * local.hour / 24),
                    "hour_cos": np.cos(2 * np.pi * local.hour / 24),
                    "dow_sin": np.sin(2 * np.pi * local.dayofweek / 7),
                    "dow_cos": np.cos(2 * np.pi * local.dayofweek / 7),
                    "is_holiday": pd.Index(local.date).isin(HOLIDAYS).astype(float)}, index=HOURS)
COV[WEATHER_COLS] = (PT.groupby("ts_hour")[WEATHER_COLS].first().reindex(HOURS)
                     .interpolate(limit_direction="both"))
COV_ARR = COV.to_numpy(np.float32)
Y = {(d, t): DEV[d][t].reindex(HOURS).to_numpy(np.float32) for d, t in PAIRS}
QCOLS = [f"q{q}" for q in QUANTILES]


def pos(ts):
    """Row of each UTC timestamp on HOURS."""
    return np.asarray((pd.DatetimeIndex(ts) - HOURS[0]) // pd.Timedelta(hours=1))


# Chronos-2 returns noise from an all-missing context (a meter offline for weeks), so a path
# needs a day of observations in the shortest context; the rest drop out of both models.
MIN_CTX = min([CONTEXT_LENGTH, *(g["context"] for g in FT_GRID)])
seen = np.array([np.isfinite(Y[d, t][p - MIN_CTX:p]).sum() >= 24 for d, t, p in zip(
    PATHS["device_id"], PATHS["target"], pos(PATHS["origin"]), strict=True)])
print("paths without history, left out:", PATHS[~seen].groupby("block").size().to_dict())
PATHS = PATHS[seen]


def covariates(a, b):
    """The covariates of rows a..b-1 of HOURS, by name."""
    return dict(zip(COV.columns, COV_ARR[a:b].T, strict=True))


def forecast(pipe, paths, context):
    """Point (median) and quantile forecasts of every (device, target, origin) row of `paths`,
    from the `context` hours before its origin, clipped at 0, one row per lead hour."""
    inputs = []
    for d, target, o in zip(paths["device_id"], paths["target"], paths["origin"], strict=True):
        p = pos([o])[0]
        assert p >= context and HOURS[p] == o  # the context is the labels before the origin
        y = Y[d, target][p - context:p]
        assert np.isfinite(y).any(), f"no history for {d} {target} at {o}"
        inputs.append({"target": y,
                       "past_covariates": covariates(p - context, p),
                       "future_covariates": covariates(p, p + HORIZON)})
    q, _ = pipe.predict_quantiles(inputs, prediction_length=HORIZON, quantile_levels=QUANTILES,
                                  batch_size=PRED_BATCH_SIZE, context_length=context)
    q = np.sort(np.maximum(0, torch.stack(q)[:, 0].float().cpu().numpy()), axis=-1)
    out = paths[KEY[:3]].loc[paths.index.repeat(HORIZON)].reset_index(drop=True)
    out["horizon"] = np.tile(np.arange(1, HORIZON + 1), len(paths))
    q = q.reshape(-1, len(QUANTILES))
    return out.assign(point=q[:, QUANTILES.index(0.5)], **dict(zip(QCOLS, q.T, strict=True)))


def finetune(end, steps, lr, context, name):
    """LoRA fine-tune of the pooled model on every series' labels before `end`; returns the
    pipeline (adapters merged) and the seconds taken. The adapter lands in models/<name>."""
    steps, lr, context = int(steps), float(lr), int(context)  # papermill's YAML reads 3e-05 as text
    inputs, p = [], pos([end])[0]
    assert HOURS[p - 1] < end <= HOURS[p]  # training labels stop before `end`
    for d, target in PAIRS:
        y = Y[d, target][:p]
        start = int(np.argmax(np.isfinite(y)))  # from the first observation
        inputs.append({"target": y[start:],
                       "past_covariates": covariates(start, p),
                       "future_covariates": {c: None for c in COV.columns}})
    set_seed(SEED)
    t0 = time.time()
    ft = PIPE.fit(inputs, prediction_length=HORIZON, finetune_mode="lora", context_length=context,
                  learning_rate=lr, num_steps=steps, batch_size=FT_BATCH_SIZE,
                  output_dir=OUT_DIR / "models" / name, remove_printer_callback=True,
                  seed=SEED, disable_tqdm=True)
    seconds = time.time() - t0
    if hasattr(ft.model, "merge_and_unload"):  # plain Chronos-2 weights for prediction
        ft = Chronos2Pipeline(model=ft.model.merge_and_unload())
    ft.model.eval()  # fit() leaves the model in train mode: dropout would stay on at predict time
    print(f"  fine-tune {name}: {steps} steps lr {lr:g} context {context} in {seconds:.0f}s"
          f" ({100 * seconds / steps:.1f}s per 100 steps)")
    return ft, seconds


def timed(pipe, paths, context, label):
    """forecast() with a timing line."""
    t0 = time.time()
    fc = forecast(pipe, paths, context)
    s, n = time.time() - t0, paths["origin"].nunique()
    print(f"  {label}: {len(paths)} paths, {n} origins in {s:.1f}s ({s / n:.2f}s per origin)")
    return fc


assert torch.cuda.is_available(), "Chronos-2 fine-tuning needs a GPU: run it with gx10_run()"
torch.cuda.set_per_process_memory_fraction(GPU_MEMORY_FRACTION)
PIPE = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cuda")
print(f"{MODEL_ID} on {torch.cuda.get_device_name(0)} | {len(PAIRS)} series,"
      f" {COV.shape[1]} covariates: {list(COV.columns)}")

In [ ]:
# Grid search on the tuning folds: every setting is fine-tuned once on the labels before
# TUNE_START, then forecasts every tuning origin. The score of a series in a fold is the mean
# of its MAE over leads 1-48, 1-96 and 1-HORIZON (each horizon counts equally); a setting
# scores the mean over folds of the mean over series. Zero-shot is scored the same way.
LEAD_CUTS = sorted({c for c in (48, 96, HORIZON) if c <= HORIZON})
TUNE_ROWS = META[META["block"] == "tune"]
tune_paths = PATHS[PATHS["block"] == "tune"]


def tune_score(fc):
    r = TUNE_ROWS.merge(fc, on=KEY)
    r["err"] = (r["point"] - r["actual"]).abs()
    per_series = sum(r[r["horizon"] <= c].groupby(["fold", "device_id", "target"])["err"].mean()
                     for c in LEAD_CUTS) / len(LEAD_CUTS)
    return per_series.groupby("fold").mean().mean()


t0 = time.time()
GRID = [dict(setting="zeroshot", context=CONTEXT_LENGTH, ft_seconds=0.0,
             score=tune_score(timed(PIPE, tune_paths, CONTEXT_LENGTH, "zero-shot tune")))]
for i, g in enumerate(FT_GRID):
    ft, seconds = finetune(T_TUNE, **g, name=f"tune{i}")
    GRID.append(dict(g, setting=f"tune{i}", ft_seconds=seconds,
                     score=tune_score(timed(ft, tune_paths, g["context"], f"tune{i}"))))
    del ft
    gc.collect()
    torch.cuda.empty_cache()
GRID = pd.DataFrame(GRID).set_index("setting")
GRID.to_csv(OUT_DIR / "grid.csv")
BEST = FT_GRID[int(GRID["score"].iloc[1:].to_numpy().argmin())]
(OUT_DIR / "best_params.json").write_text(json.dumps(BEST, indent=2))
TUNE_SECONDS = time.time() - t0
print(f"tuned in {TUNE_SECONDS:.0f}s | best {BEST}"
      f" | peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
display(GRID.round(4))

In [ ]:
# The best setting is fine-tuned before TEST_START for the test folds, and before CUTOFF for
# the replay, where it stays frozen. Zero-shot forecasts the same paths.
t0, parts = time.time(), []
for block, end in [("test", T_TEST), ("replay", CUT)]:
    paths = PATHS[PATHS["block"] == block]
    ft, _ = finetune(end, **BEST, name=block)
    parts += [timed(PIPE, paths, CONTEXT_LENGTH, f"zero-shot {block}").assign(
                  model="chronos2_zeroshot"),
              timed(ft, paths, BEST["context"], f"fine-tuned {block}").assign(model="chronos2")]
    del ft
    gc.collect()
    torch.cuda.empty_cache()
RES = META[META["block"] != "tune"].merge(pd.concat(parts), on=KEY)
EVAL_SECONDS = time.time() - t0

COLS = ["model", "scope", "device_id", "target", "origin", "horizon", "ts_hour", "actual", "point",
        *QCOLS]
# The dtypes of 03's replay.parquet: plain strings, ns timestamps, float64 values.
out = RES.assign(scope="pooled").astype(
    {**dict.fromkeys(["model", "scope", "device_id", "target"], object),
     **dict.fromkeys(["fold", "origin", "ts_hour"], "datetime64[ns, UTC]"),
     **dict.fromkeys(["point", *QCOLS], "float64")})
out[out["block"] == "test"][[*COLS[:4], "fold", *COLS[4:]]].to_parquet(
    OUT_DIR / "test.parquet", index=False)
out[out["block"] == "replay"][COLS].to_parquet(OUT_DIR / "replay.parquet", index=False)
print(f"test + replay in {EVAL_SECONDS:.0f}s | adapters in {OUT_DIR / 'models'}")

In [ ]:
BANDS = ["1-8", "9-24", "25-48", "49-96", "97-168", "all"]
QA = np.array(QUANTILES)
RES["band"] = pd.cut(RES["horizon"], [0, 8, 24, 48, 96, 168], labels=BANDS[:-1]).astype(str)
RES["export_day"] = RES["is_daylight"].eq(1) & RES["target"].eq("grid_export")


def metrics(g):
    """Point, interval and naive-baseline scores of a group of rows, in kWh/h."""
    y, err = g["actual"], g["point"] - g["actual"]
    diff = y.to_numpy()[:, None] - g[QCOLS].to_numpy()
    m = {"mae": err.abs().mean(), "rmse": np.sqrt((err**2).mean())}
    m |= {f"mae_{c}": (g[c] - y).abs().mean() for c in ["naive168", "naive24"]}
    m["skill168"] = 1 - m["mae"] / m["mae_naive168"] if m["mae_naive168"] > 0.01 else np.nan
    m["pinball"] = np.maximum(QA * diff, (QA - 1) * diff).mean()
    m["cov50"] = y.between(g["q0.25"], g["q0.75"]).mean()
    m["cov80"] = y.between(g["q0.1"], g["q0.9"]).mean()
    m["mae_daylight"] = err[g["export_day"]].abs().mean()  # export only
    return pd.Series(m)


both = pd.concat([RES, RES.assign(band="all")])
both["band"] = pd.Categorical(both["band"], BANDS)
SUMMARY = both.groupby(["block", "model", "target", "band"], observed=True).apply(
    metrics, include_groups=False)
PER_DEVICE = RES.groupby(["block", "model", "target", "device_id"]).apply(
    metrics, include_groups=False)
PER_DEVICE = PER_DEVICE[["mae", "mae_naive168", "skill168"]].unstack("block")
SUMMARY.to_csv(OUT_DIR / "summary.csv")
PER_DEVICE.to_csv(OUT_DIR / "per_device.csv")
print(f"tuned in {TUNE_SECONDS:.0f}s, test + replay in {EVAL_SECONDS:.0f}s | kWh/h")
display(SUMMARY.xs("all", level="band").round(4))
display(PER_DEVICE.round(4))

In [ ]:
if LOG_MLFLOW:
    import mlflow

    default_uri = f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}"
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", default_uri))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params({**PARAMS, "host": os.uname().nodename,
                           "tune_seconds": round(TUNE_SECONDS), "eval_seconds": round(EVAL_SECONDS),
                           **{f"best.{k}": v for k, v in BEST.items()}})
        mlflow.log_metrics({f"tune.score.{s}": v for s, v in GRID["score"].items()})
        mlflow.log_metrics({f"{block}.{model}.{metric}.{t}.{band}": v
                            for (block, model, t, band), r in SUMMARY.iterrows()
                            for metric, v in r.items() if pd.notna(v)})
        mlflow.log_artifacts(str(OUT_DIR))
    print("logged to", mlflow.get_tracking_uri())

In [ ]:
# Run this notebook on the GX10 (the only GPU): save the notebook, run the first two code cells
# and this one, then call gx10_run(); keyword arguments override the top cell, e.g.
# gx10_run(DEVICES=["c2g-9FFB8A0D0"], FT_GRID=[dict(steps=50, lr=1e-4, context=512)]).
# gx10_status() shows running jobs, gx10_pull() brings runs/nb05 and its MLflow runs back.
# PyPI torch on aarch64 has no CUDA, so the job runs in the GPU sandbox container (NVIDIA vLLM
# image, venv /w/.venv-nv with chronos-forecasting, peft, papermill), which mounts
# ~/chronos-sandbox at /w and is capped at 24 GB; it is kept apart from ~/celine-forecasting.
# Each job folder holds this notebook, 02's outputs, the executed copy and the log; results
# and the MLflow db land in ~/chronos-sandbox/runs/nb05.
import shlex
import subprocess

GX10_HOST = os.environ.get("GX10_HOST")                # user@host of the box, set it in your shell
GX10_DIR = "chronos-sandbox"                        # on the box, relative to the home folder
GX10_CONTAINER = "chronos-sandbox-nv"
GX10_MIN_FREE_GB = 15                               # the box freezes if memory runs out


def gx10_run(**overrides):
    """Copy this notebook and 02's outputs to a job folder on the box, then run it there in the
    GPU container, with `overrides` (parameter name: value) on top of the top cell."""
    if not GX10_HOST:
        raise RuntimeError("set GX10_HOST (user@host) in the environment before using the GX10 helpers")
    free = subprocess.run(["ssh", GX10_HOST, "free -g | awk '/Mem:/ {print $7}'"], check=True,
                          capture_output=True, text=True).stdout
    assert int(free) >= GX10_MIN_FREE_GB, f"only {free.strip()} GB available on the GX10"
    p = {**PARAMS, **overrides}
    p["RUN_NAME"] = p["RUN_NAME"] or f"chronos2-h{p['HORIZON']}-{time.strftime('%Y%m%d-%H%M%S')}"
    job = p["RUN_NAME"]
    jobdir = f"runs/nb05/jobs/{job}"
    subprocess.run(["ssh", GX10_HOST, f"mkdir -p {GX10_DIR}/{jobdir}/data"], check=True)
    subprocess.run(["rsync", "-az", str(NB_DIR / "05_chronos2.ipynb"),
                    f"{GX10_HOST}:{GX10_DIR}/{jobdir}/"], check=True)
    inputs = [str(DATA_DIR / f) for f in ["processed_hourly.parquet", "device_quality.csv"]]
    subprocess.run(["rsync", "-az", *inputs, f"{GX10_HOST}:{GX10_DIR}/{jobdir}/data/"], check=True)
    w = f"/w/{jobdir}"
    remote = (
        # only docker goes to the background
        f"cd {GX10_DIR}/{jobdir} || exit 1; "
        f"nohup docker exec -e HOME=/tmp -e MLFLOW_TRACKING_URI=sqlite:////w/runs/nb05/mlflow.db "
        f"{GX10_CONTAINER} /w/.venv-nv/bin/papermill {w}/05_chronos2.ipynb "
        f"{w}/05_chronos2.out.ipynb --cwd {w} -y {shlex.quote(json.dumps(p))} "
        "> log.txt 2>&1 < /dev/null &")
    subprocess.run(["ssh", GX10_HOST, remote], check=True)
    print(f"started nb05 job {job} on {GX10_HOST}")


def gx10_status():
    """Running nb05 jobs with the last lines of their logs, and the box's free memory."""
    remote = (f"cd {GX10_DIR}/runs/nb05/jobs; "
              f"for j in $(docker exec {GX10_CONTAINER} pgrep -af papermill "
              "| grep -o 'jobs/[^/]*' | sort -u | cut -d/ -f2); do echo \"running: $j\"; "
              "tail -c 600 $j/log.txt | tr '\\r' '\\n' | tail -n 3; done; "
              "echo latest: $(ls -td */ | head -1); free -g | head -2")
    print(subprocess.run(["ssh", GX10_HOST, remote], capture_output=True, text=True).stdout)


def gx10_pull():
    """Copy runs/nb05 from the box to runs/gx10/nb05/ here (without the job inputs), then its
    MLflow runs to our tracking server."""
    local = REPO_ROOT / "runs" / "gx10" / "nb05"
    subprocess.run(["rsync", "-az", "--exclude", "jobs/*/data/",
                    f"{GX10_HOST}:{GX10_DIR}/runs/nb05/", str(local)], check=True)
    gx10_to_mlflow(local / "mlflow.db")


def gx10_to_mlflow(db):
    """Copy the box's runs (params, metrics, tags) to MLflow here, skipping those already copied."""
    from mlflow.entities import Metric, Param
    from mlflow.tracking import MlflowClient

    src = MlflowClient(f"sqlite:///{db}")
    uri = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}")
    dst = MlflowClient(uri)
    copied = 0
    for exp in src.search_experiments():
        target = dst.get_experiment_by_name(exp.name)
        exp_id = target.experiment_id if target else dst.create_experiment(exp.name)
        # The box's run id travels as a tag, so pulling twice never copies a run twice.
        done = {r.data.tags.get("gx10.run_id")
                for r in dst.search_runs([exp_id], max_results=50000)}
        for run in src.search_runs([exp.experiment_id], max_results=50000):
            if run.info.run_id in done:
                continue
            tags = {**run.data.tags, "gx10.run_id": run.info.run_id, "gx10.host": GX10_HOST}
            new = dst.create_run(exp_id, start_time=run.info.start_time, tags=tags,
                                 run_name=run.info.run_name).info.run_id
            metrics = [Metric(m.key, m.value, m.timestamp, m.step) for k in run.data.metrics
                       for m in src.get_metric_history(run.info.run_id, k)]
            params = [Param(k, v) for k, v in run.data.params.items()]
            for i in range(0, len(metrics), 1000):  # log_batch caps: 1000 metrics, 100 params
                dst.log_batch(new, metrics=metrics[i:i + 1000])
            for i in range(0, len(params), 100):
                dst.log_batch(new, params=params[i:i + 100])
            dst.set_terminated(new, run.info.status, run.info.end_time)
            copied += 1
    print(f"copied {copied} new run(s) from {db} to {uri}")

In [ ]:
# gx10_run()
# gx10_status()
# gx10_pull()